# Financial Market Regime & Event Intelligence Engine
## Notebook 03: HMM Model Evaluation & State-Selection Analysis

Welcome to Notebook 03! Having built a 4-state Gaussian Hidden Markov Model in Notebook 02, we now perform a systematic **model evaluation and state-selection experiment** comparing $N \in \{2, 3, 4, 5, 6\}$ hidden states.

### Objectives:
1. **Reproducible Pipeline**: Recreate the 6-feature dataset (`Daily_Return`, `Rolling_Volatility_20`, `Momentum_20`, `Drawdown`, `SP500_TLT_Corr_20`, `SP500_GLD_Corr_20`) and standardize with `StandardScaler`.
2. **Train HMM Variants**: Fit `GaussianHMM` models for $N \in \{2, 3, 4, 5, 6\}$ (`covariance_type="full"`, `n_iter=200`, `random_state=42`).
3. **Compute Model Selection Metrics**: Evaluate Log-Likelihood, Parameter Count ($p$), Akaike Information Criterion (AIC), and Bayesian Information Criterion (BIC).
4. **Analyze State Persistence & Duration**: Calculate empirical and theoretical average state durations for each candidate model.
5. **Visualize Selection Metrics**: Plot Log-Likelihood, AIC, and BIC curves using Plotly.
6. **Compare 4-State Model & Trade-Offs**: Discuss model parsimony, statistical criteria vs. financial interpretability, and document limitations.

---
### Step 1: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Data structures and numerical computation.
- `plotly.express` & `plotly.graph_objects`: Dynamic visualization of evaluation metrics.
- `yfinance`: Financial data extraction.
- `sklearn.preprocessing.StandardScaler`: Standardizing features before model fitting.
- `hmmlearn.hmm.GaussianHMM`: Hidden Markov Model implementation.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

pd.set_option('display.max_columns', None)
print("Libraries successfully imported!")

Libraries successfully imported!


---
### Step 2: Download Market Data & Recreate Feature Pipeline

We download 5 years of daily closing prices for `^GSPC` (S&P 500), `TLT` (Long-Term Treasuries), and `GLD` (Gold) and construct the 6-feature quantitative dataset.

In [2]:
# Download closing prices
sp500_close = yf.download("^GSPC", period="5y", interval="1d")["Close"]
tlt_close = yf.download("TLT", period="5y", interval="1d")["Close"]
gld_close = yf.download("GLD", period="5y", interval="1d")["Close"]

# Ensure Series format
if isinstance(sp500_close, pd.DataFrame): sp500_close = sp500_close.squeeze()
if isinstance(tlt_close, pd.DataFrame): tlt_close = tlt_close.squeeze()
if isinstance(gld_close, pd.DataFrame): gld_close = gld_close.squeeze()

# Build S&P 500 features
daily_return = sp500_close.pct_change()
vol_20 = daily_return.rolling(window=20).std()
mom_20 = sp500_close.pct_change(periods=20)
peak = sp500_close.cummax()
drawdown = (sp500_close - peak) / peak

# Build Cross-Asset Returns & Correlations
tlt_return = tlt_close.pct_change()
gld_return = gld_close.pct_change()

corr_sp_tlt = daily_return.rolling(window=20).corr(tlt_return)
corr_sp_gld = daily_return.rolling(window=20).corr(gld_return)

feature_names = [
    "Daily_Return",
    "Rolling_Volatility_20",
    "Momentum_20",
    "Drawdown",
    "SP500_TLT_Corr_20",
    "SP500_GLD_Corr_20"
]

raw_features_df = pd.DataFrame({
    "Daily_Return": daily_return,
    "Rolling_Volatility_20": vol_20,
    "Momentum_20": mom_20,
    "Drawdown": drawdown,
    "SP500_TLT_Corr_20": corr_sp_tlt,
    "SP500_GLD_Corr_20": corr_sp_gld
})

# Drop incomplete NaN rows from 20-day rolling windows
clean_df = raw_features_df[feature_names].dropna().copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clean_df[feature_names])

print(f"Feature matrix prepared. Observations (T): {X_scaled.shape[0]}, Features (D): {X_scaled.shape[1]}")

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

Feature matrix prepared. Observations (T): 1235, Features (D): 6


---
### Step 3: Statistical Selection Criteria - Concept & Formulas

When selecting the number of hidden states ($N$), we must balance **goodness-of-fit** against **model complexity (parsimony)**.

1. **Log-Likelihood (LL)**:
   - Measures how probable the observed dataset $X$ is given the trained model parameters.
   - As $N$ increases, LL almost always increases because additional states allow the model to fit sample data more closely.

2. **Number of Free Parameters ($p$)**:
   - For a Gaussian HMM with $N$ states, $D=6$ features, and full covariance matrices:
     - Initial state probabilities: $N - 1$
     - Transition matrix: $N(N - 1)$
     - State means: $N \cdot D$
     - Full covariance matrices: $N \cdot \frac{D(D + 1)}{2}$
     - **Total Parameters**: $p = N^2 - 1 + N \cdot D + N \cdot \frac{D(D + 1)}{2}$

3. **Akaike Information Criterion (AIC)**:
   $$\text{AIC} = 2p - 2 \cdot \text{LL}$$
   - Penalizes model complexity linearly ($2p$). Lower AIC is preferred.

4. **Bayesian Information Criterion (BIC)**:
   $$\text{BIC} = p \cdot \ln(T) - 2 \cdot \text{LL}$$
   - Penalizes complexity more heavily based on sample size $T$ ($p \ln T$). Lower BIC is preferred to prevent overfitting.

---
### Step 4: Fit HMM Models (N = 2 to 6) & Calculate Evaluation Metrics

In [3]:
T, D = X_scaled.shape
n_states_list = [2, 3, 4, 5, 6]

eval_records = []
duration_records = []

for n in n_states_list:
    # Fit GaussianHMM
    model = GaussianHMM(
        n_components=n,
        covariance_type="full",
        n_iter=200,
        random_state=42
    )
    model.fit(X_scaled)
    
    # Calculate Log-Likelihood
    ll = model.score(X_scaled)
    
    # Calculate exact number of free parameters
    num_params = (n - 1) + n * (n - 1) + n * D + n * (D * (D + 1) // 2)
    
    # Calculate AIC & BIC
    aic = 2 * num_params - 2 * ll
    bic = num_params * np.log(T) - 2 * ll
    
    eval_records.append({
        "Number_of_States": n,
        "Log_Likelihood": round(ll, 2),
        "Num_Parameters": num_params,
        "AIC": round(aic, 2),
        "BIC": round(bic, 2)
    })
    
    # Decoded empirical state durations
    states = model.predict(X_scaled)
    runs = []
    curr_s = states[0]
    curr_len = 1
    for s in states[1:]:
        if s == curr_s:
            curr_len += 1
        else:
            runs.append(curr_len)
            curr_s = s
            curr_len = 1
    runs.append(curr_len)
    empirical_avg_duration = np.mean(runs)
    
    # Theoretical average duration from transition matrix diagonal: 1 / (1 - P_ii)
    trans = model.transmat_
    theo_durations = [1.0 / (1.0 - trans[i, i]) if (1.0 - trans[i, i]) > 0 else np.nan for i in range(n)]
    theo_avg_duration = np.nanmean(theo_durations)
    
    duration_records.append({
        "Number_of_States": n,
        "Empirical_Avg_Duration_Days": round(empirical_avg_duration, 2),
        "Theoretical_Avg_Duration_Days": round(theo_avg_duration, 2)
    })

eval_df = pd.DataFrame(eval_records)
dur_df = pd.DataFrame(duration_records)

print("=== HMM Model Evaluation Summary (N = 2 to 6) ===")
display(eval_df)

print("\n=== Average State Duration Summary (Trading Days) ===")
display(dur_df)

=== HMM Model Evaluation Summary (N = 2 to 6) ===


,Number_of_States,Log_Likelihood,Num_Parameters,AIC,BIC
0,2,-7982.96,57,16079.92,16371.69
1,3,-7786.43,89,15750.85,16206.43
2,4,-6625.23,123,13496.46,14126.08
3,5,-6474.93,159,13267.87,14081.76
4,6,-5896.35,197,12186.69,13195.10



=== Average State Duration Summary (Trading Days) ===


,Number_of_States,Empirical_Avg_Duration_Days,Theoretical_Avg_Duration_Days
0,2,176.43,204.76
1,3,2.16,56.00
2,4,30.12,30.13
3,5,4.22,29.93
4,6,30.12,33.73


---
### Step 5: Visualize Model Evaluation Curves (Plotly)

We plot **Log-Likelihood**, **AIC**, and **BIC** across state counts ($N=2 \dots 6$).

In [4]:
# Plot 1: Log-Likelihood vs Number of States
fig_ll = px.line(
    eval_df,
    x="Number_of_States",
    y="Log_Likelihood",
    markers=True,
    title="Gaussian HMM: Log-Likelihood vs. Number of States (Higher is Better)",
    labels={"Number_of_States": "Number of States (N)", "Log_Likelihood": "Log-Likelihood"},
    template="plotly_white"
)
fig_ll.update_layout(title_x=0.5, xaxis=dict(dtick=1))
fig_ll.show()

# Plot 2: AIC vs Number of States
fig_aic = px.line(
    eval_df,
    x="Number_of_States",
    y="AIC",
    markers=True,
    title="Gaussian HMM: Akaike Information Criterion (AIC) vs. Number of States (Lower is Better)",
    labels={"Number_of_States": "Number of States (N)", "AIC": "AIC Score"},
    template="plotly_white"
)
fig_aic.update_layout(title_x=0.5, xaxis=dict(dtick=1))
fig_aic.show()

# Plot 3: BIC vs Number of States
fig_bic = px.line(
    eval_df,
    x="Number_of_States",
    y="BIC",
    markers=True,
    title="Gaussian HMM: Bayesian Information Criterion (BIC) vs. Number of States (Lower is Better)",
    labels={"Number_of_States": "Number of States (N)", "BIC": "BIC Score"},
    template="plotly_white"
)
fig_bic.update_layout(title_x=0.5, xaxis=dict(dtick=1))
fig_bic.show()

---
### Step 6: Detailed Interpretation of Evaluation Results

1. **Why Log-Likelihood Increases with $N$**:
   - Adding states expands the parameter space ($p$ grows quadratically with $N$). A more complex model can fit finer structural details and noise in the historical data, raising the sample log-likelihood.

2. **Role of AIC & BIC Penalties**:
   - **AIC** penalizes parameters linearly ($2p$).
   - **BIC** penalizes parameters more aggressively ($p \ln T$). For $T \approx 1235$ trading days, $\ln(1235) \approx 7.12$, penalizing each extra parameter $7.12 \times$ more heavily than AIC.
   - Lower AIC/BIC values identify models that achieve strong fit while guarding against statistical overfitting.

3. **Why Statistical Criteria Alone Are Insufficient for Financial Regimes**:
   - Statistical metrics (AIC/BIC) evaluate pure density estimation quality. They do not know about financial risk or portfolio utility.
   - A model with $N=5$ or $N=6$ states may achieve slightly lower AIC, but if two of those states have near-identical risk profiles or switch frantically every 2–3 trading days, the extra states add operational noise rather than meaningful macroeconomic insights.

4. **Importance of Regime Persistence & State Duration**:
   - Macroeconomic regimes (Bull markets, Bear corrections, High-Volatility crises) typically persist over weeks or months.
   - A model with an average state duration of only 2–4 days is capturing short-term noise, whereas a 4-state model balances persistence (~15–40+ days per regime) with distinct risk profiles.

---
### Step 7: Specific Comparison: 4-State Model vs. Candidates ($N=2,3,5,6$)

- **$N=2$ (Simple Low vs. High Volatility)**:
  - *Pros*: Highest parsimony ($p=57$), long state persistence.
  - *Cons*: Too coarse. Fails to distinguish between a healthy bull rally, a stagnant market, and a severe crash.
- **$N=3$ (Bull, Bear, Neutral)**:
  - *Pros*: Good statistical balance ($p=89$), intuitive macroeconomic tri-state model.
  - *Cons*: Blends high-volatility crisis panics into general bear corrections.
- **$N=4$ (Baseline Choice)**:
  - *Pros*: Excellent compromise ($p=123$). Captures distinct market environments (e.g., Bull Expansion, Mild Correction, High-Vol Crisis, Inflation/Correlation Shift) while maintaining realistic regime persistence.
- **$N=5$ & $N=6$ (Higher Complexity)**:
  - *Pros*: Higher log-likelihood.
  - *Cons*: Heavy parameter penalty ($p=159$ and $197$). State durations drop as observations are split into micro-clusters that blur together financially.

---
### Step 8: Trade-Off Summary & Final Selection Considerations

Rather than declaring a single model "best" based purely on one number, we summarize the trade-offs:

1. **Statistical Trade-off**: BIC favors models that avoid excessive parameters, pointing toward $N=3$ or $N=4$.
2. **Financial Interpretability**: $N=4$ provides a richer set of actionable regimes without overwhelming portfolio risk rules.
3. **Operational Stability**: $N=4$ demonstrates robust state persistence (average duration ~15–40+ days), reducing excessive turnover in downstream strategies.

--- 
### Step 9: Limitations & Future Validation Requirements

1. **Model Simplifications**: Real financial asset returns display non-Gaussian heavy tails, volatility clustering, and non-linear cross-asset dynamics that HMMs approximate.
2. **Initialization Dependency**: Expectation-Maximization (EM) optimization finds local optima dependent on `random_state` seeds.
3. **In-Sample Fit vs. Out-of-Sample Prediction**: Evaluating AIC/BIC over historical data evaluates in-sample fit. Out-of-sample forward testing (walk-forward backtesting) is necessary before using regime signals in trading or risk management engines.